# Suite SNP — Snapshots: a store written as two transactions, and read back

A store's snapshot ([Serialization](../../docs/SERIALIZATION.md), Snapshots) is written by
`Plain.ToPlain(store).Snapshot()` as two transactions (object identity 101 in [CAP-127: snapshots made of transactions](../../../mbse-basis/docs/capabilities/CAP-127.md#decided)): the first upserts its
schemas, each a symbol found by its name, a schema object with no name of its own defined where it is linked; the
second, its data, every object its singletons reach a symbol, each entry written once. `Plain.FromPlain(store).Snapshot(
snapshot)` executes them in order. Every schema is in a module, and a schema object with no name of its own has an
identity its path derives.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects typed
loosely, so callbacks are annotated `(t: any)`; ints are `bigint`s, plain mappings are `Map`s, and a snapshot written
again is compared as its JSON text.

In [ ]:
import { Builders as B, JSON as Json, Paths, Plain, Proxies, Stores } from "@mbse/schemas/Framework";
import { DecodeError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, entries_in, equal, given, map, raises, same, spec_of } from "./support.js";

/** crm.Person, who knows other persons since a year, and a singleton crm.Registry listing persons. */
function people(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm.people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Knows: S.OfRelation, Lists: S.OfRelation, Registry: S.OfObject })
    .spec([
      (_: any) => _.Person.name("crm.Person").ref(true).properties({
        given_name: S.OfStr, tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) }).adjacencies({
        acquaintances: (a: any) => a.relation([_.Knows]).me("knower") }).adjacencies([
        (a: any) => a.relation([_.Knows]).me("known")]),
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }).uniques([["since"], ["knower", "known"]]),
      (_: any) => _.Lists.name("crm.Lists").links([(t: any) => t.name("registry"), (t: any) => t.name("person")]),
      (_: any) => _.Registry.name("crm.Registry").ref(true).singleton("the.Registry").properties({ title: S.OfStr }).adjacencies({
        people: (a: any) => a.relation([_.Lists]).me("registry") }),
    ])]))]);
}

/** A schema object's path, from the root module ".", as the names of its steps joined by dots. */
function dotted(path: any): string {
  const names: string[] = [];
  let step = path.get("next");
  while (step.has("entry")) {
    names.push(step.get("entry").get("properties").get("name"));
    step = step.get("entry").get("next");
  }
  return names.join(".") || ".";
}

/** The names of the schema objects a snapshot's first transaction finds, in order: null, the root module's. */
function schemasIn(snapshot: any[]): (string | null)[] {
  const pattern = snapshot[0].get("mutations")[0].get("upsert").get("exists")[0];
  return pattern.get("spec").flatMap((item: any) => [...item.values()].map((members: any) => given(members).get("name") ?? null));
}

/** A snapshot as its JSON text, to compare two snapshots as Python compares them with `==`, in order. */
const text = (snapshot: unknown) => Json.dumps(snapshot as never);

## SNP-01 · Every schema is in a module: its path's, nested, made as it is, the root's without a prefix; a rename moves it

In [ ]:
{
  const store: any = new B.Store();
  const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
  store.transaction(people);
  const Person = store.registered("crm.Person");
  const [Loose, Order] = store.transaction(() => { // modules as the builders make them
    const Loose = store.Schemas.OfObject((o: any) => o.name("Loose"));
    const Order = store.Schemas.OfObject((o: any) => o.name("crm.sales.Order"));
    return [Loose, Order];
  });
  const [root, crm, sales] = [store.module(), store.module("crm"), store.module("crm.sales")];
  assert(root.name === null && same([...root.schemas.keys()], ["crm", "Loose"]));
  assert(equal(Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(root)), map({ singleton: ".", next: { target: {} } }))); // "." alone
  assert(same([...crm.schemas.keys()], ["Person", "Knows", "Lists", "Registry", "sales"]) && crm.schemas.get("Person") === Person);
  assert(same([...new B.ModuleBuilder(crm, false, store).schemas].slice(0, 2).map((e: any) => e.name.get()), ["Person", "Knows"])); // its entries' names
  assert(sales.schemas.get("Order") === Order && crm.schemas.get("sales") === sales && store.held(Paths.of(sales)) === sales);
  store.transaction(() => {
    store.Schemas.OfObject(Order, (o: any) => o.name("crm.Order")); // renamed: moved
  });
  assert(!sales.schemas.has("Order") && crm.schemas.get("Order") === Order);
  store.transaction(() => {
    store.Schemas.OfModule(sales, (m: any) => m.name("crm.shop")); // a module renamed
  });
  assert(store.module("crm.shop") === sales);
  const [bad] = store.transaction(() => {
    const bad = store.Schemas.OfObject((o: any) => o.name("not a path"));
    return [bad];
  });
  assert([root, crm].every((module) => [...module.schemas.values()].every((held) => held !== bad))); // validation reports its name
  raises(LookupError, () => Paths.of(bad), "the name 'not a path' is not a path: its parts are names, joined by dots");
  assert(store.framework(store.Schemas.OfObject.as_schema()) && store.framework(store.Schemas.OfInt.as_schema()) && !store.framework(Person));
  assert(store.held(Paths.of(Person)) === Person && store.holds(Person) && !store.holds(new (Int.constructor as any)()));
  const nowhere = map({ singleton: ".", next: { entry: { i: "schemas", o: "", properties: { name: "crm" }, next: { entry: {
    i: "schemas", o: "", properties: { name: "Person" }, next: { entry: { i: "properties", o: "", properties: {
      name: "nope" }, next: { target: {} } } } } } } } });
  assert(store.held(nowhere) === null); // a member its owner does not hold
  const [extra] = store.transaction(() => {
    const extra = new B.ModuleBuilder(null, true, store).name("extra").schemas([Loose]).create(); // a schema linked with no name
    return [extra];
  });
  assert(same([...extra.schemas.keys()], ["Loose"]));
  const held = schemasIn(Plain.ToPlain(store).Snapshot() as any[]);
  assert(held.includes("extra") && held.at(-1) === "not a path"); // a module linking a schema; a schema no module holds, written last
}

## SNP-02 · Paths: a member's its owner's then its name; what has no name of its own, and what it holds, has none

In [ ]:
{
  const store: any = new B.Store();
  const Int = store.Schemas.OfInt.as_schema();
  store.transaction(people);
  const [Person, Knows] = [store.registered("crm.Person"), store.registered("crm.Knows")];
  store.transaction(() => {
    /** A schema object's path, shown as `dotted`; null where it has none. */
    const shown = (schemaObject: unknown): string | null => {
      try {
        return dotted(Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(schemaObject)));
      } catch (error) {
        if (error instanceof LookupError) return null;
        throw error;
      }
    };
    const tags = Person.properties.get("tags");
    assert(shown(tags) === "crm.Person.tags" && shown(tags.schema) === null); // an unnamed schema: written where it is linked
    assert(shown(Person.adjacencies.get("acquaintances")) === "crm.Person.acquaintances");
    assert(shown(Person.anonymous[0]) === null); // an anonymous adjacency has no name of its own
    assert(shown(Knows.entry.links.get("knower")) === "crm.Knows.knower");
    const Unnamed = store.Schemas.OfObject((o: any) => o.name("crm.Unnamed").properties((adv: any) => adv.insert((e: any) => e.property((p: any) => p))));
    assert(shown(Unnamed.properties.get("")) === null); // a member its entry does not name
    const Kinds = store.Schemas.OfPattern((p: any) => p.name("crm.Kinds").symbols([(t: any) => t.name("x")]).variables([(t: any) => t.name("x")]));
    assert(shown(Kinds.variables.get("x")) === "crm.Kinds.x" && shown(Kinds.symbols.get("x")) === "crm.Kinds.x"); // shown alike, through other adjacencies
    assert(!Paths.same(Paths.of(Kinds.variables.get("x")), Paths.of(Kinds.symbols.get("x"))));
    const Shape = store.Schemas.OfUnion((u: any) => u.name("crm.Shape").branches([
      (t: any) => t.name("a").schema([Int]),
      (t: any) => t.name("b").schema([store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("x").schema([Int])]))]),
    ]));
    assert(shown(Shape.branches[1].schema) === null && shown(Shape.branches[1].schema.properties.get("x")) === null); // nor what it holds
  });
}

## SNP-03 · A snapshot: two transactions, its schemas, each found by its name, then its data, the singletons first; each entry once

In [ ]:
/** crm.sales.Order, in a module of its own within crm's. */
function orders(txn: any): any {
  return txn.uid("crm.orders/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Order: txn.Schemas.OfObject })
    .spec([(_: any) => _.Order.name("crm.sales.Order").ref(true)])]))]);
}

{
  const store: any = new B.Store();
  store.transaction(people);
  store.transaction(orders);
  store.transaction(() => {
    const bob = store.crm.Person((c: any) => c.given_name("Bob"));
    const ada = store.crm.Person((c: any) => (
      c
        .given_name("Ada")
        .tags(["x", "y"])
        .acquaintances((adv: any) => (
          adv
            .insert((e: any) => e.known(bob).since(2020n))
        ))
    ));
    store.crm.Registry(store.singleton("the.Registry"), (r: any) => r.title("main").people((a: any) => a.insert(
      (e: any) => e.person(ada)).insert((e: any) => e.person(bob)))); // a snapshot writes only the data the singletons reach
  });
  const snapshot = Plain.ToPlain(store).Snapshot() as any[];
  assert(snapshot.length === 2 && snapshot.every((t) => same([...t.keys()], ["mutations"]))); // its schemas', then its data's
  assert(same(schemasIn(snapshot), [null, "crm", "crm.Person", "crm.Knows", "crm.Lists", "crm.Registry", "crm.sales",
    "crm.sales.Order"])); // its modules, then what each holds, in order, nested; the store's own never
  assert(equal(spec_of(snapshot, "0")[0], map({ as_property: { name: "name", mutations: [{ clear: {} }] } }))); // the root, which has none
  const person = spec_of(snapshot, "crm.Person");
  assert(equal(given(person), map({ name: "crm.Person", ref: true })));
  assert(same(entries_in(person, "properties").map((e) => e.get("name")), ["given_name", "tags"])); // each defined where it is linked
  const tags = entries_in(person, "properties")[1]!.get("property").get("def");
  assert(same(entries_in(tags.get("mutations"), "schema").map((e) => e.get("schema").get("def").get("schema")), ["Schemas.OfContainer"])); // unnamed
  assert(same(entries_in(person, "adjacencies").map((e) => e.get("adjacency").get("def").get("schema")),
    ["Schemas.OfAdjacency", "Schemas.OfAdjacency"])); // its anonymous one too
  assert(equal(given(spec_of(snapshot, "crm.Knows")).get("uniques"), [["since"], ["knower", "known"]]));
  const data = snapshot[1].get("mutations")[0].get("upsert").get("exists")[0].get("symbols");
  assert(same([...data.keys()], ["0", "1", "2"])); // the singleton, then what it reaches, in the order met
  assert(equal(given(spec_of(snapshot, "0", 1)), map({ title: "main" }))
    && equal(given(spec_of(snapshot, "1", 1)), map({ given_name: "Ada", tags: ["x", "y"] }))); // untyped payloads
  assert(equal(entries_in(spec_of(snapshot, "1", 1), "acquaintances"), [map({ known: { ref: { symbol: "2" } }, since: 2020n })]));
  assert(entries_in(spec_of(snapshot, "2", 1), "acquaintances").length === 0); // Bob's end of it is not written again
}

## SNP-04 · A snapshot read back into another store, and written again the same; its data held by the caller

In [ ]:
/** crm.Named, a pattern over crm.Person, which `people` installed: its spec is written with it, as recorded. */
function naming(txn: any): any {
  return txn.uid("crm.named/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Named: txn.Schemas.OfPattern })
    .spec([(_: any) => _.Named.name("crm.Named").symbols([(t: any) => t.name("a").schema([txn.crm.Person])]).spec([
      (_: any) => _.a.given_name("Ada")])])]))]);
}

{
  const store: any = new B.Store();
  store.transaction(people);
  const [Person, Knows] = [store.registered("crm.Person"), store.registered("crm.Knows")];
  store.transaction(() => {
    const bob = store.crm.Person((c: any) => c.given_name("Bob"));
    const ada = store.crm.Person((c: any) => c.given_name("Ada").acquaintances((adv: any) => adv.insert((e: any) => e.known(bob))));
    store.crm.Registry(store.singleton("the.Registry"), (r: any) => r.title("main").people((a: any) => a.insert(
      (e: any) => e.person(ada)).insert((e: any) => e.person(bob)))); // a snapshot writes only the data the singletons reach
  });
  const snapshot = Plain.ToPlain(store).Snapshot();
  const back: any = new B.Store();
  const made: any[] = Plain.FromPlain(back).Snapshot(snapshot);
  assert(text(Plain.ToPlain(back).Snapshot()) === text(snapshot));
  assert(back.registered("crm.Person") !== Person && same([...back.registered("crm.Person").properties.keys()], ["given_name", "tags"]));
  const clauses = (uniques: readonly ReadonlySet<string>[]) => uniques.map((unique) => [...unique].sort()); // as Python compares frozensets
  assert(equal(clauses(back.registered("crm.Knows").uniques), clauses(Knows.uniques)) && back.singleton("the.Registry").title.get() === "main");
  const peopleBack = made.filter((m: any) => m.as_schema() === back.registered("crm.Person"));
  assert(made[0] === back.singleton("the.Registry") && same(peopleBack.map((p: any) => String(p.given_name)).sort(), ["Ada", "Bob"]));
  const again: any[] = Plain.FromPlain(back).Snapshot(snapshot); // read again: the counted persons made anew, the rest found
  assert(again[0] === made[0] && again.slice(1).every((a, k) => a !== made[k + 1]));
  assert(same(again.slice(1).map((p: any) => String(p.given_name)).sort(), ["Ada", "Bob"]) && [...back.singleton("the.Registry").people].length === 4);
  store.transaction(naming);
  const named = Plain.ToPlain(store).Snapshot() as any[];
  const written = map({ spec: { items: [{ a: [{ as_property: { name: "given_name", mutations: [{ set: { value: "Ada" } }] } }] }] } });
  assert(equal(spec_of(named, "crm.Named").at(-1), written)); // after its members
  Plain.FromPlain(back).Snapshot(named);
  assert(equal(back.registered("crm.Named").spec, written.get("spec").get("items")));
  back.transaction(() => {
    back.Schemas.OfPattern(back.registered("crm.Named"), (p: any) => p.spec([(_: any) => _.a.given_name("Bob")])); // added to it
  });
  Plain.FromPlain(back).Snapshot(named); // read again: the spec it gives, in place of what the pattern held
  assert(equal(back.registered("crm.Named").spec, written.get("spec").get("items")));
  Plain.FromPlain(back).Snapshot(named); // and again: it holds, so it is left as it is
  assert(equal(back.registered("crm.Named").spec, written.get("spec").get("items")));
  const malformed = Json.loads(text(named).replace('{"spec": {"items": [{"a"', '{"spec": {"items": [{"a": 1, "b"'));
  const unshaped: any = Json.loads(text(named));
  spec_of(unshaped, "crm.Named").at(-1).set("spec", "x"); // not a mapping of its items
  for (const bad of [malformed, unshaped]) {
    raises(DecodeError, () => Plain.FromPlain(new B.Store()).Snapshot(bad),
      "expected a pattern's spec: items, a list of mappings of symbols to their objects' mutations");
  }
  store.transaction(() => { // modules are written too, each found by its path: an empty one, and what describes one
    store.Schemas.OfModule((m: any) => m.name("crm.empty"));
    store.Schemas.OfModule(store.module("crm"), (m: any) => m.description("Customers"));
    store.Schemas.OfModule(store.module(), (m: any) => m.description("Everything")); // the root module's, which has no name
  });
  const moduled = Plain.ToPlain(store).Snapshot();
  const fresh: any = new B.Store();
  Plain.FromPlain(fresh).Snapshot(moduled);
  assert(same([...fresh.module("crm").schemas.keys()], [...store.module("crm").schemas.keys()])); // in the order it holds them
  assert(fresh.module("crm").description === "Customers" && fresh.module().description === "Everything");
  assert(text(Plain.ToPlain(fresh).Snapshot()) === text(moduled));
}

## SNP-05 · What a snapshot refuses: a store that is not a Builders.Store, a schema the store does not hold, a malformed snapshot

In [ ]:
/** Pairs, a relation whose entries are a set: one empty clause, which a snapshot keeps. */
function pairs(txn: any): any {
  return txn.uid("pairs/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Pairs: txn.Schemas.OfRelation })
    .spec([(_: any) => _.Pairs.name("Pairs").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([[]])])]))]);
}

/** crm.Foreign, whose n is an int. */
function foreign(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm.foreign/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Foreign: S.OfObject })
    .spec([(_: any) => _.Foreign.name("crm.Foreign").properties({ n: S.OfInt })])]))]);
}

/** crm.Anonymous, whose first link's end at crm.Person is anonymous, its second's named `as_b`. */
function anonymous(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm.anonymous/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Anonymous: S.OfRelation, Person: S.OfObject })
    .spec([
      (_: any) => _.Anonymous.name("crm.Anonymous").links([(t: any) => t.name("a"), (t: any) => t.name("b")]),
      (_: any) => _.Person.name("crm.Person").adjacencies([(a: any) => a.relation([_.Anonymous]).me("a")]).adjacencies({
        as_b: (a: any) => a.relation([_.Anonymous]).me("b") }),
    ])]))]);
}

{
  raises(TypeError, () => Plain.ToPlain(new Stores.Combined(new B.Store())).Snapshot(), "a snapshot is of a Builders.Store"); // stores combined have no snapshot of their own
  raises(TypeError, () => Plain.FromPlain(new Stores.Combined(new B.Store())).Snapshot([]), "a snapshot is read into a Builders.Store");
  const setlike: any = new B.Store();
  setlike.transaction(pairs);
  const copied: any = new B.Store();
  Plain.FromPlain(copied).Snapshot(Plain.ToPlain(setlike).Snapshot());
  const uniques = copied.Pairs.as_schema().uniques;
  assert(uniques.length === 1 && uniques[0].size === 0); // read back, not dropped
  let store: any = new B.Store();
  store.transaction(foreign);
  const Foreign = store.registered("crm.Foreign");
  new B.OfProperty.Builder(Foreign.properties.get("n")).schema([new B.OfObject.Data()]).update(); // behind the store's back: no store builds it
  raises(ValueError, () => Plain.ToPlain(store).Snapshot(), "crm.Foreign.n, Schemas.HasSchema, links _ObjectData, a schema the store does not hold");
  store = new B.Store();
  store.transaction(people);
  store.transaction(anonymous);
  store.transaction(() => {
    const bob = store.crm.Person((c: any) => c.given_name("Bob"));
    const ada = store.crm.Person((c: any) => c.given_name("Ada").as_b((adv: any) => adv.insert((e: any) => e.a(bob))));
    store.crm.Registry(store.singleton("the.Registry"), (r: any) => r.people((a: any) => a.insert((e: any) => e.person(ada))));
  });
  const snapshot = Plain.ToPlain(store).Snapshot();
  const back: any = new B.Store();
  Plain.FromPlain(back).Snapshot(snapshot); // an entry added through its second link's adjacency: its first is anonymous
  assert(text(Plain.ToPlain(back).Snapshot()) === text(snapshot));
  for (const [bad, says] of [
    [map({}), "$: expected a snapshot, a list of transactions"],
    [[[]], "$[0]: expected an object of Mutations.OfTransaction, a mapping of its members"],
    [[map({ exists: {} })], "$[0].exists: 'exists' is not a member of Mutations.OfTransaction"],
    [Json.loads(text(snapshot).replace('"given_name"', '"age"')), "crm.Person has no property 'given_name'"], // its schema's, renamed
    [Json.loads(text(snapshot).replaceAll('"Person"', '"Nobody"')),
      '$[1].mutations[0].upsert.exists[0].symbols["1"].schema[0].$ref: no object at this path'], // its data's schema, by path
  ] as [unknown, string][]) {
    raises(DecodeError, () => Plain.FromPlain(new B.Store()).Snapshot(bad), says);
  }
  const ignored: Plain.Ignored[] = []; // a member a schema the store was made holding lacks is an extension's it lacks: ignored, not refused
  const extended: any = Json.loads(text(snapshot));
  const items = extended[0].get("mutations")[0].get("upsert").get("exists")[0].get("spec");
  const i = items.findIndex((item: any) => [...item.values()].some((members: any) => given(members).get("name") === "crm.Person"));
  const symbol = [...items[i].keys()][0];
  items[i].get(symbol).push(map({ as_property: { name: "nope", mutations: [{ set: { value: 1n } }] } }));
  Plain.FromPlain(new B.Store()).Snapshot(extended, ignored);
  assert(ignored.length === 1 && ignored[0]!.path === `$[0].mutations[0].upsert.exists[0].spec[${i}]["${symbol}"][4].as_property`
    && ignored[0]!.reason === "Schemas.OfObject has no property 'nope'");
  const specced: any = Json.loads(text(snapshot));
  specced[0].get("mutations")[0].get("upsert").get("exists")[0].get("spec")[i].get(symbol).push(map({ spec: { items: [] } }));
  raises(DecodeError, () => Plain.FromPlain(new B.Store()).Snapshot(specced), "Schemas.OfObject has no spec"); // only a pattern has one
}